# PointNet simplificado sobre ModelNet10

Notebook unico de principio a fin, pensado para **Google Colab con GPU T4**
gratuita: descarga y explora los datos, preprocesa, entrena y evalua el
modelo, todo en la misma sesion.

(Antes esto estaba en dos notebooks separados - exploracion y
entrenamiento/evaluacion - pero Colab gratuito solo permite una sesion con
GPU activa a la vez, y abrir el segundo notebook cerraba la sesion del
primero, perdiendo la cache de datos ya descargada. Un unico notebook evita
ese problema.)

Contenido:
1. Setup (clonar repo, dependencias)
2. Descargar ModelNet10
3. Exploracion: conteo de muestras por clase
4. Preprocesar (muestreo + normalizacion) y cachear
5. Sanity check de la normalizacion
6. Visualizar nubes de puntos de ejemplo
7. Entrenamiento
8. Curvas de aprendizaje
9. Evaluacion en test: accuracy, classification report, matriz de confusion
10. Visualizacion de predicciones concretas

## 1. Setup

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.isdir("3d_classifier"):
        !git clone https://github.com/marsaliborra/3d_classifier.git
    %cd 3d_classifier
    !git pull
    !pip install -q trimesh
    print("Ejecutando en Colab.")
else:
    print("Ejecutando localmente (asumo que ya estoy en la raiz del repo o en notebooks/).")

In [ ]:
import sys
from pathlib import Path

import torch

# Si se ejecuta localmente desde notebooks/, src/ esta un nivel por encima.
# En Colab ya estamos en la raiz del repo tras el %cd de la celda anterior.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("AVISO: sin GPU, el entrenamiento sera lento. Runtime > Change runtime type > T4 GPU.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import CLASSES, download_modelnet10, list_off_files, get_dataset_arrays

DATA_ROOT = REPO_ROOT / "data"
N_POINTS = 1024  # mismo numero de puntos que usa el paper original de PointNet

## 2. Descargar ModelNet10

`download_modelnet10` es idempotente: si `data/ModelNet10` ya existe (por
ejemplo, porque ya se ejecuto esta celda antes en la misma sesion), no
vuelve a descargar los ~450MB.

In [ ]:
dataset_dir = download_modelnet10(root=str(DATA_ROOT))
dataset_dir

## 3. Exploracion: conteo de muestras por clase

Antes de entrenar nada, conviene saber si el dataset esta balanceado entre
clases. Si una clase tuviera muchas menos muestras que las demas, la
accuracy global podria ser enganosa, y habria que tenerlo en cuenta al leer
la matriz de confusion mas adelante.

In [ ]:
train_samples = list_off_files(dataset_dir, "train")
test_samples = list_off_files(dataset_dir, "test")

train_counts = [sum(1 for _, c in train_samples if c == cls) for cls in CLASSES]
test_counts = [sum(1 for _, c in test_samples if c == cls) for cls in CLASSES]

for split_name, counts in [("train", train_counts), ("test", test_counts)]:
    print(f"--- {split_name} (total={sum(counts)}) ---")
    for cls, n in zip(CLASSES, counts):
        print(f"  {cls:12s}: {n}")

x = np.arange(len(CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width / 2, train_counts, width, label="train")
ax.bar(x + width / 2, test_counts, width, label="test")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_ylabel("numero de muestras")
ax.set_title("ModelNet10: muestras por clase y split")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Preprocesar (muestreo + normalizacion) y cachear

`get_dataset_arrays` hace todo el trabajo pesado: por cada malla `.off`
muestrea `N_POINTS` puntos sobre su superficie y normaliza la nube
resultante (centrado + escalado a esfera unitaria). El resultado se guarda
en `data/modelnet10_<split>_<n>pts.npz` para no repetir este paso - que
puede tardar varios minutos sobre ~4000 mallas - cada vez que se reinicia
el runtime.

In [ ]:
train_points, train_labels = get_dataset_arrays(dataset_dir, "train", n_points=N_POINTS, cache_dir=str(DATA_ROOT))
test_points, test_labels = get_dataset_arrays(dataset_dir, "test", n_points=N_POINTS, cache_dir=str(DATA_ROOT))

print("train_points:", train_points.shape, train_points.dtype)
print("train_labels:", train_labels.shape, train_labels.dtype)
print("test_points: ", test_points.shape)
print("test_labels: ", test_labels.shape)

## 5. Sanity check de la normalizacion

Para cada nube, el centroide deberia estar en (0,0,0) (dentro de error de
float32) y la distancia maxima al origen deberia ser exactamente 1.0. Si
esto fallara, significaria que el modelo esta viendo objetos a escalas y
posiciones inconsistentes entre si, lo cual romperia la premisa de que la
red aprende FORMA y no posicion/escala.

In [ ]:
sample_idx = 0
cloud = train_points[sample_idx]

centroid = cloud.mean(axis=0)
max_radius = np.linalg.norm(cloud, axis=1).max()

print(f"Clase: {CLASSES[train_labels[sample_idx]]}")
print(f"Centroide (deberia ser ~[0,0,0]): {centroid}")
print(f"Radio maximo (deberia ser ~1.0): {max_radius:.4f}")

## 6. Visualizar nubes de puntos de ejemplo

Inspeccion visual de 4 muestras aleatorias del set de entrenamiento, para
confirmar "a ojo" que las formas son reconocibles despues del muestreo y la
normalizacion.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (necesario para projection='3d')

rng = np.random.default_rng(42)
sample_indices = rng.choice(len(train_points), size=4, replace=False)

fig = plt.figure(figsize=(16, 4))
for i, idx in enumerate(sample_indices):
    ax = fig.add_subplot(1, 4, i + 1, projection="3d")
    cloud = train_points[idx]
    ax.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], s=2, alpha=0.6)
    ax.set_title(CLASSES[train_labels[idx]])
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 7. Entrenamiento

`train()` reutiliza la cache de datos ya generada arriba, entrena
`SimplifiedPointNet` con `CrossEntropyLoss` + `Adam`, y guarda el mejor
checkpoint (`best_model.pt`, segun accuracy de test) y el historial de
metricas por epoca (`history.json`) en `outputs/checkpoints/`.

Con 30 epocas, batch 32 y una T4, deberia tardar del orden de varios
minutos.

In [ ]:
from src.train import train

model, history = train()

## 8. Curvas de aprendizaje

Loss y accuracy de train vs test por epoca. Si train sigue mejorando
mientras test se estanca o empeora, es la senal de overfitting - con un
dataset y modelo tan pequenos, vale la pena revisarlo antes de fiarse solo
de la accuracy final.

In [ ]:
from src.evaluate import plot_training_curves

plot_training_curves(history, REPO_ROOT / "outputs" / "figures" / "training_curves.png")

## 9. Evaluacion en test

Carga el mejor checkpoint guardado durante el entrenamiento y calcula:
- accuracy global en el test set
- un `classification_report` de sklearn (precision/recall/F1 por clase -
  util para ver si el modelo falla de forma pareja o se concentra en
  confundir un par de clases geometricamente parecidas, p.ej. desk/table)
- la matriz de confusion (normalizada por fila, con seaborn)

In [ ]:
from sklearn.metrics import classification_report

from src.evaluate import CHECKPOINT_DIR, FIGURES_DIR, load_trained_model, plot_confusion_matrix, predict_all

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model = load_trained_model(CHECKPOINT_DIR / "best_model.pt", device)
predictions = predict_all(best_model, test_points, device)

accuracy = (predictions == test_labels).mean()
print(f"Accuracy en test: {accuracy:.4f} ({(predictions == test_labels).sum()}/{len(test_labels)})\n")
print(classification_report(test_labels, predictions, target_names=CLASSES))

In [ ]:
plot_confusion_matrix(test_labels, predictions, FIGURES_DIR / "confusion_matrix.png")

## 10. Ejemplos de predicciones

4 nubes de puntos aleatorias del test set con su etiqueta real y la
prediccion del modelo (verde = acierto, rojo = error).

In [ ]:
from src.evaluate import plot_example_predictions

plot_example_predictions(test_points, test_labels, predictions, FIGURES_DIR / "example_predictions.png")

## Siguiente paso

Con la accuracy, la matriz de confusion y los ejemplos ya generados, toca
volcar los resultados reales al `README.md` del repo (sustituyendo los
`TODO`): que funciono, que confunde el modelo y por que, y que se haria
distinto con mas tiempo.